# 01 — Historical archive

The competition exposes a historical archive so you can experiment **offline**:
one gzip-JSONL file per `event_type` × `quarter`, each line a past event with its
disclosure. This notebook:

1. Browses the archive manifest (`GET /archive`)
2. Downloads and **caches** the files locally (`data/archive/`, gitignored)
3. Loads them into a pandas DataFrame
4. Inspects a single event and its disclosure
5. Looks at the **earnings preview** — the second disclosure item recent events carry
6. Reproduces the baseline scoring regressions

With a live key you pull the real archive; without one, you load the small bundled
sample so the mechanics are still clear.

> Download URLs are short-lived and signed. The helper refreshes an expired URL
> automatically (via `GET /archive/{event_type}/{quarter}`) when you pass it a
> client.

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

from examples import Client, load_config
from examples.archive import download_archive, load_archive
from examples.frames import manifest_frame
from examples.preview import (
    earnings_preview_from_disclosure,
    preview_display_markdown,
    preview_title,
    previews_frame,
)
from examples.summary import facts_from_disclosure

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "sample").is_dir())
ARCHIVE_DIR = REPO / "data" / "archive"  # gitignored download cache
SAMPLE_DIR = REPO / "data" / "sample"  # tiny bundled stand-in

config = load_config()
print("Mode:", "LIVE" if config.is_live else "SAMPLE (no EM_API_KEY — using bundled data)")

## 1. Browse the manifest

Each row is one downloadable file — what event type and quarter it covers, how
many events it holds, and how big it is. (The signed URLs are omitted from this
view.)

In [ ]:
manifest = None
if config.is_live:
    with Client.from_env() as client:
        manifest = client.archive_manifest()
    display(manifest_frame(manifest))
else:
    print("Sample mode: no live manifest. We'll load the bundled sample archive below.")

## 2. Download & cache

`download_archive` writes each file into `data/archive/`, **skipping** any file
already cached at the size the manifest reports — so re-running is cheap. Narrow
the pull with `only=`, e.g. `only=lambda f: f.quarter >= "2026Q1"`.

A quarter whose manifest row says `sealed = False` is still filling in: its file
grows as events are scored and disclosures are attached, so its size changes and
the cache check re-downloads it. Sealed quarters never change.

In [ ]:
if config.is_live:
    with Client.from_env() as client:
        paths = download_archive(
            manifest,
            ARCHIVE_DIR,
            client=client,  # lets it refresh expired URLs
            # only=lambda f: f.event_type == "EARNINGS_RELEASE",
        )
    source = ARCHIVE_DIR
    print(f"{len(paths)} file(s) cached in {ARCHIVE_DIR}")
else:
    source = SAMPLE_DIR
    print(f"Sample mode: reading the bundled archive in {SAMPLE_DIR}")

## 3. Load into pandas

`load_archive` reads every `*.jsonl.gz` under a directory (or a single file) into
one DataFrame, adding `event_type` and `quarter` provenance columns from each
filename.

In [ ]:
df = load_archive(source)
print(f"Loaded {len(df):,} events")
print("Columns:", list(df.columns))
df.head()

## 4. Inspect one event

Pull a single row and read its focal assets and disclosure. The bundled sample
inlines disclosures under a `disclosure` field; the live archive carries additional
fields — inspect `df.columns` to see what a given dataset provides.

`disclosure.items[]` can hold **two** items: the ten earnings-call facts
(`kind="facts"`, the subject of notebook 02) and, for recent events, the earnings
preview (`id="earnings-preview"`, `kind="text"`, section 5). `facts_from_disclosure`
and `earnings_preview_from_disclosure` look each one up and return `[]` / `None`
when it is absent — never assume a position in the list.

In [ ]:
row = df.iloc[0]
record = row.to_dict()

focal = record.get("focal_assets")
tickers = [a["identifier_value"] for a in (focal if isinstance(focal, list) else [])]
print(f"event_type={row['event_type']}  quarter={row['quarter']}  tickers={tickers}")
print(f"event_datetime={record.get('event_datetime')}\n")

for fact in facts_from_disclosure(record):
    print("  -", fact)

preview = earnings_preview_from_disclosure(record)
if preview is None:
    print("\nNo earnings preview on this event (expected before 2026Q3 — see section 5).")
else:
    print(f"\nEarnings preview attached: {preview_title(preview)!r}, {len(preview):,} chars")

## 5. The earnings preview (2026Q3 onward)

From 2026Q3, an event's disclosure may carry a second item alongside the facts:
the **earnings preview**, an agent-written research note assembled from public
sources *before* the release — the market's ex ante information set, in one
markdown string. It is optional, so count coverage rather than assuming it.

Going forward a disseminated preview is assembled before the event's CAR1
measurement window opens (its `knowledge_cutoff`). The 2026Q3 archive is the
exception — those notes were produced ahead of each release but not always
before the cutoff — so treat them as illustrative. Notebook 03 covers how the
preview is built and reads several in full; here we just check coverage and
render one.

In [ ]:
records = df.to_dict("records")
coverage = previews_frame(records)

by_quarter = coverage.groupby("quarter")["has_preview"].agg(events="size", with_preview="sum")
display(by_quarter)

with_preview = coverage[coverage["has_preview"]]
if with_preview.empty:
    print("No previews in this dataset.")
else:
    print(f"Earliest event with a preview: {with_preview['event_datetime'].min()}")

In [ ]:
if not with_preview.empty:
    # Prefer NVDA when present; otherwise the most recent event with a preview.
    pick = with_preview[with_preview["tickers"] == "NVDA"]
    pick = (pick if not pick.empty else with_preview).sort_values("event_datetime").iloc[-1]
    record = next(r for r in records if r["event_id"] == pick["event_id"])
    preview = earnings_preview_from_disclosure(record)
    print(f"{pick['tickers']}  {pick['event_id']}  event={pick['event_datetime']}")
    print(f"knowledge_cutoff={pick['knowledge_cutoff']}  {len(preview or ''):,} chars\n")
    display(Markdown(preview_display_markdown(preview or "")))

## 6. Reproduce the baseline scoring regressions

`examples.scoring` ports the competition's **exact** scoring transform, so you can
(i) reproduce the leaderboard's numbers offline and (ii) use it in your own
analyses and optimizations. Scoring runs *within a period* (a quarter): the
realized one-day cumulative abnormal return (`car1`) of every outcome is
percentile-ranked into the dependent variable `y = r_abn`, and the earnings
surprise is percentile-ranked into the `s` regressor. Each reference baseline
already submitted a predicted percentile `ĝ`.

We reproduce **Table 3** of Koijen & Levy (WP) on the disseminated data: `y`
regressed on five specifications —

1. surprise only (`s`) — the naive benchmark,
2. Gemini only (`ĝ`),
3. GPT-5 nano only (`ĝ`),
4. surprise + Gemini,
5. surprise + GPT-5 nano.

This section needs the **live** archive (the bundled sample carries no returns or
baseline predictions). We use `2025Q4`, the sealed quarter behind the paper's
table. (The paper also reports bootstrapped R² intervals; we show point estimates
and standard errors.)

In [ ]:
from examples.scoring import BASELINE_LABELS, add_percentiles, outcomes_frame

PERIOD = "2025Q4"  # sealed quarter matching Koijen & Levy (WP), Table 3

needed = {"event_returns", "metrics", "baseline_predictions"}
in_period = df["quarter"] == PERIOD
have_scoring_data = (
    needed <= set(df.columns)
    and in_period.any()
    and df.loc[in_period, "event_returns"].notna().any()  # the sample's mixed frame has NaN here
)

if have_scoring_data:
    # Percentiles are ranked WITHIN the period, so restrict to the quarter first.
    records = df[df["quarter"] == PERIOD].to_dict("records")
    scored = add_percentiles(outcomes_frame(records))
    models = list(BASELINE_LABELS.values())  # ["Gemini 2.5 Flash-Lite", "GPT-5 nano"]
    # The paper's single sample: outcomes with y, a surprise, and both baselines.
    fit_df = scored.dropna(subset=["y", "surprise_pct", *models]).copy()
    print(f"{PERIOD}: {len(scored):,} scored outcomes | fit sample N = {len(fit_df):,}")
else:
    print(f"Sample mode: section 6 needs the live archive for {PERIOD}. Skipping.")

In [ ]:
if have_scoring_data:
    import pandas as pd
    import statsmodels.api as sm
    from stargazer.stargazer import Stargazer

    gemini, gpt = models  # display names, in BASELINE_LABELS order
    S, G = "s (surprise pctile)", "ĝ (model pctile)"

    def fit(cols):
        # cols: {display_name -> source column}; shared names let stargazer stack rows.
        X = sm.add_constant(pd.DataFrame({name: fit_df[src] for name, src in cols.items()}))
        return sm.OLS(fit_df["y"].to_numpy(), X).fit()

    fits = [
        fit({S: "surprise_pct"}),  # (1) Baseline: surprise only
        fit({G: gemini}),  # (2) Gemini only
        fit({G: gpt}),  # (3) GPT-5 nano only
        fit({S: "surprise_pct", G: gemini}),  # (4) surprise + Gemini
        fit({S: "surprise_pct", G: gpt}),  # (5) surprise + GPT-5 nano
    ]

    star = Stargazer(fits)
    star.title(f"Transcript summaries — Koijen & Levy (WP) Table 3, reproduced on {PERIOD}")
    star.custom_columns(["Baseline", gemini, gpt, gemini, gpt], [1, 1, 1, 1, 1])
    star.covariate_order(["const", S, G])
    star.rename_covariates({"const": "Intercept"})
    star.dependent_variable_name("r_abn  (percentiled CAR1)")
    star.show_model_numbers(True)
    display(star)

## Where to go from here

The archive gives you the **inputs** — events and their disclosures. To turn this
into a backtest you'd:

1. Pair each event with the realized next-day reaction for its focal asset, from
   your own market-data source.
2. Convert that reaction into a percentile rank in `[0, 1]` **across the
   quarter's cross-section** of scored event outcomes — a rank against all the
   period's events, not against the asset's own history. That's the target the
   competition scores; `examples.scoring.percentile_ranks` is the exact
   transform (see section 6).
3. Score your model's predicted percentile against it offline, and iterate.

For the ex ante side of each event — the earnings preview — see
[`03_earnings_previews.ipynb`](03_earnings_previews.ipynb).

When your model is ready, move it into a deployed webhook handler (see the starter
repos) — that's what receives live events and submits predictions. These notebooks
deliberately stop short of `POST /predictions`.